# Data Wrangling: Join, Combine, and Reshape

In many applications, data may be spread across a number of files or databases, or be arranged in a form that is not convenient to analyze. This chapter focuses on tools ot help combine, join, and rearrange data. 

# Hierarchical Indexing 

*Hierarchical indexing* is an import feature of pandas that enables you to have multiple (two or more) index *levels* on an axis. Another way of thinking about it is that it provides a way for you to work with higher dimensional data in a lower dimensional form. Let's start with a simple example:

Create a Series with a list of lists (or arrays) as the index:

In [2]:
import pandas as pd 
import numpy as np 

In [3]:
data = pd.Series(np.random.uniform(size=9),
                index=[["a", "a", "a", "b", "b", "c", "c", "d", "d"],
                        [1, 2, 3, 1, 3, 1, 2, 2, 3]])

print(data)

a  1    0.146419
   2    0.406413
   3    0.997796
b  1    0.917360
   3    0.372317
c  1    0.738575
   2    0.196916
d  2    0.601001
   3    0.116672
dtype: float64


What you're seeing is a prettified view of a Series with a `` MultiIndex`` as its index. The "gaps" in the index display mean "use the label directly above":

In [4]:
data.index

MultiIndex([('a', 1),
            ('a', 2),
            ('a', 3),
            ('b', 1),
            ('b', 3),
            ('c', 1),
            ('c', 2),
            ('d', 2),
            ('d', 3)],
           )

With a hierarchially indexed object, so-called *partial* indexing is possible, enabling you to concisely select subsets of the data:

In [5]:
data["b"]

1    0.917360
3    0.372317
dtype: float64

In [6]:
data["b":"c"]

b  1    0.917360
   3    0.372317
c  1    0.738575
   2    0.196916
dtype: float64

In [7]:
data.loc[["b", "d"]]

b  1    0.917360
   3    0.372317
d  2    0.601001
   3    0.116672
dtype: float64

Selection is even possible from an "inner" level. Here I select all of the values having the value 2 from the second index level:

In [8]:
data.loc[:, 2]

a    0.406413
c    0.196916
d    0.601001
dtype: float64

Hierarchial indexing plays an important role in reshaping data and in group-based operations like forming a pivot table. For example, you can rearrange this data into a DataFrame using its ``unstack`` method:

In [9]:
data.unstack()

,1,2,3
a,0.146419,0.406413,0.997796
b,0.917360,NaN,0.372317
c,0.738575,0.196916,NaN
d,NaN,0.601001,0.116672


The inverse operation of ``unstack`` is ``stack``:

In [10]:
data.unstack().stack()

a  1    0.146419
   2    0.406413
   3    0.997796
b  1    0.917360
   3    0.372317
c  1    0.738575
   2    0.196916
d  2    0.601001
   3    0.116672
dtype: float64

``stack`` and ``unstack`` with be explored in more detail later. 

With a DataFrame, either axis can have a hierarchial index:

In [11]:
frame = pd.DataFrame(np.arange(12).reshape((4, 3)),
                    index =[["a", "a", "b", "b"], [1, 2, 1, 2]],
                    columns=[["Ohio", "Ohio", "Colorado"],
                            ["Green", "Red", "Green"]])

frame

Ohio     Colorado
    Green Red    Green
a 1     0   1        2
  2     3   4        5
b 1     6   7        8
  2     9  10       11

In [12]:
print(frame)

     Ohio     Colorado
    Green Red    Green
a 1     0   1        2
  2     3   4        5
b 1     6   7        8
  2     9  10       11


The hierarchial levels can have names (as strings or any Python objects). If so, these will show up in the console output:

In [13]:
frame.index.names = ["key1", "key2"]

frame.columns.names = ["state", "color"]

frame

state      Ohio     Colorado
color     Green Red    Green
key1 key2                   
a    1        0   1        2
     2        3   4        5
b    1        6   7        8
     2        9  10       11

These names supersede the ``name`` attribute, which is used only with single-level indexes. 

You can see how many levels an index has by accessing its ``nlevels`` attribute:

In [14]:
frame.index.nlevels

2

With partial column indexing, you can similarly select groups of columns:

In [15]:
frame["Ohio"]

color      Green  Red
key1 key2            
a    1         0    1
     2         3    4
b    1         6    7
     2         9   10

A ``MultiIndex`` can be created by itself and then reused; the columns in the preceding DataFrame with level names could also be created like this:

In [16]:
pd.MultiIndex.from_arrays([["Ohio", "Ohio", "Colorado"],
                        ["Green", "Red", "Green"]], 
                        names = ["state", "color"])

MultiIndex([(    'Ohio', 'Green'),
            (    'Ohio',   'Red'),
            ('Colorado', 'Green')],
           names=['state', 'color'])

## Reordering and Sorting Levels 

At times you may need to rearrange the order of the levels on an axis or sort the data by the values in one specific level. The ``swaplevel`` method takes two level numbers or names and returns a new object with the levels interchanged (but the data is otherwise unaltered):

In [17]:
frame.swaplevel("key1", "key2")

state      Ohio     Colorado
color     Green Red    Green
key2 key1                   
1    a        0   1        2
2    a        3   4        5
1    b        6   7        8
2    b        9  10       11

``sort_index`` by default sorts the data lexicographically using all the index levels, but you can choose to use only a single level or a subset of levels to sort by passing the ``level`` argument. For example:

In [18]:
frame.sort_index(level=1)

state      Ohio     Colorado
color     Green Red    Green
key1 key2                   
a    1        0   1        2
b    1        6   7        8
a    2        3   4        5
b    2        9  10       11

In [19]:
frame.swaplevel(0, 1).sort_index(level=0)

state      Ohio     Colorado
color     Green Red    Green
key2 key1                   
1    a        0   1        2
     b        6   7        8
2    a        3   4        5
     b        9  10       11

### Summary Statistics by Level

Many descriptive and summary statistics on DataFrame and Series have a ``level`` option in which you can specify the level you want to aggregate by on a particular axis. Consider the above DataFrame; we can go aggregate by level on either the rows or columns, like so:

In [20]:
frame.groupby(level="key2").sum()

state  Ohio     Colorado
color Green Red    Green
key2                    
1         6   8       10
2        12  14       16

In [21]:
frame.T.groupby(level="color").sum()

key1   a      b    
key2   1  2   1   2
color              
Green  2  8  14  20
Red    1  4   7  10

### Indexing with a DataFrame's columns

It's not unusual to want to use one or more columns from a DataFrame as the row index; alternatively, you may wish to move the row index into the DataFrame's columns. Here's an example DataFrame:

In [22]:
frame = pd.DataFrame({"a": range(7), "b": range(7, 0, -1),
                    "c": ["one", "one", "one", "two", 'two', "two", "two"],
                    "d": [0, 1, 2, 0, 1, 2, 3]})

frame

,a,b,c,d
0,0,7,one,0
1,1,6,one,1
2,2,5,one,2
3,3,4,two,0
4,4,3,two,1
5,5,2,two,2
6,6,1,two,3


DataFrame's ``set_index`` function will create a new DataFrame using one or more of its columns as the index:

In [23]:
frame2 = frame.set_index(["c", "d"])

frame2

a  b
c   d      
one 0  0  7
    1  1  6
    2  2  5
two 0  3  4
    1  4  3
    2  5  2
    3  6  1

By default, the columns are removed from the DataFrame, though you can leave them in by passing ``drop = False`` to ``set_index``:

In [24]:
frame.set_index(["c", "d"], drop=False)


a  b    c  d
c   d              
one 0  0  7  one  0
    1  1  6  one  1
    2  2  5  one  2
two 0  3  4  two  0
    1  4  3  two  1
    2  5  2  two  2
    3  6  1  two  3

``reset_index``, on the other hand, does the opposite of ``set_index``; the hierarchial index levels are moved into the columns:

In [25]:
frame2.reset_index()

,c,d,a,b
0,one,0,0,7
1,one,1,1,6
2,one,2,2,5
3,two,0,3,4
4,two,1,4,3
5,two,2,5,2
6,two,3,6,1


# Combining and Merging Datasets

Data contained in pandas objects can be combined in a number of ways:

``pandas.merge`` 

Connect rows in DataFrames based on one or more keys. This will be familiar to users of SQL or other relational databases, as it implements database *join* operations. 

``pandas.concat``

Concatenate or "stack" objects together along an axis. 

``combine_first``

Splice together overlapping data to fill in missing values in one object with values from another. 

## Database-Style DataFrame Joins

*Merge* or *join* operations combine datasets by linking rows using one or more keys. These operations are particularly important in relational databases. The ``pandas.merge`` function in pandas is the main entry point for using these algorithms on your data. 

In [26]:
df1 = pd.DataFrame({"key": ["b", "b", "a", "c", "a", "a", "b"],
                    "data1": pd.Series(range(7), dtype="Int64")})
df2 = pd.DataFrame({"key": ["a", "b", "d"],
                    "data2": pd.Series(range(3), dtype="Int64")})

df1


,key,data1
0,b,0
1,b,1
2,a,2
3,c,3
4,a,4
5,a,5
6,b,6


In [27]:
df2

,key,data2
0,a,0
1,b,1
2,d,2


Here I am using pandas's ``Int64`` extension type for nullable integers, discussed earlier. 

This is an example of a *many-to-one* join; the data in ``df1`` has multiple rows labeled ``a`` and ``b``, whereas ``df2`` has only one row for each value in the ``key`` column. Calling ``pandas.merge`` with these objects, we obtain: 

In [28]:
pd.merge(df1, df2)

,key,data1,data2
0,b,0,1
1,b,1,1
2,a,2,0
3,a,4,0
4,a,5,0
5,b,6,1


Note that I didn't specify which column to join on. If that information is not specified, ``pandas.merge`` uses the overlapping column names as the keys. It's a good practice to specify explicitly, though:

In [29]:
pd.merge(df1, df2, on="key")

,key,data1,data2
0,b,0,1
1,b,1,1
2,a,2,0
3,a,4,0
4,a,5,0
5,b,6,1


In general, the order of column output in ``pandas.merge`` operations is specified. 

If the column names are different in each object, you can specify them separately:

In [30]:
df3 = pd.DataFrame({"lkey": ["b", "b", "a", "c", "a", "a", "b"],
                    "data1": pd.Series(range(7), dtype="Int64")})
df4 = pd.DataFrame({"rkey": ["a", "b", "d"],
                    "data2": pd.Series(range(3), dtype="Int64")})

pd.merge(df3, df4, left_on="lkey", right_on="rkey")

,lkey,data1,rkey,data2
0,b,0,b,1
1,b,1,b,1
2,a,2,a,0
3,a,4,a,0
4,a,5,a,0
5,b,6,b,1


You may notice that the "c" and "d" values and associated data are missing from the result. By default, ``pandas.merge`` does an "``inner``" join; the keys in the result are the intersection, or the common set found in both tables. Other possible options are "``left``", "``right``", and "``outer``". The ``outer`` join takes the union of the keys combining the effect of applying both left and right joins:

In [31]:
pd.merge(df1, df2, how="outer")

,key,data1,data2
0,a,2,0
1,a,4,0
2,a,5,0
3,b,0,1
4,b,1,1
5,b,6,1
6,c,3,<NA>
7,d,<NA>,2


    Table 8.1: Different join types with the how argument

| Option | Behavior |
|---|---|
| `how="inner"` | Use only the key combinations observed in both tables |
| `how="left"` | Use all key combinations found in the left table |
| `how+"right"` | Use all key combinations found in the right table |
| `how="outer"` | Use all key combinations observed in both tables together |

*Many-to-many* merges for the Cartesian product of the matching keys. Here's an example:

In [32]:
df1 = pd.DataFrame({"key": ["b", "b", "a", "c", "a", "b"],
                    "data1": pd.Series(range(6), dtype="Int64")})
df2 = pd.DataFrame({"key": ["a", "b", "a", "b", "d"],
                    "data2": pd.Series(range(5), dtype="Int64")})

In [33]:
pd.merge(df1, df2, on="key", how="left")

,key,data1,data2
0,b,0,1
1,b,0,3
2,b,1,1
3,b,1,3
4,a,2,0
5,a,2,2
6,c,3,<NA>
7,a,4,0
8,a,4,2
9,b,5,1


Since there were three "``b``" rows in the left DataFrame and two in the right one, there are six "``b``" rows in the result. The join method passed to the ``how`` keyword argument affect only the distinct key values appearing in the result:

In [34]:
pd.merge(df1, df2, how="inner")

,key,data1,data2
0,b,0,1
1,b,0,3
2,b,1,1
3,b,1,3
4,a,2,0
5,a,2,2
6,a,4,0
7,a,4,2
8,b,5,1
9,b,5,3


To merge with multiple keys, pass a list of columns names:

In [35]:
left = pd.DataFrame({"key1": ["foo", "foo", "bar"],
                     "key2": ["one", "two", "one"],
                     "lval": pd.Series([1, 2, 3], dtype='Int64')})
right = pd.DataFrame({"key1": ["foo", "foo", "bar", "bar"],
                      "key2": ["one", "one", "one", "two"],
                      "rval": pd.Series([4, 5, 6, 7], dtype='Int64')})

pd.merge(left, right, on=["key1", "key2"], how="outer")

,key1,key2,lval,rval
0,bar,one,3,6
1,bar,two,<NA>,7
2,foo,one,1,4
3,foo,one,1,5
4,foo,two,2,<NA>


## Combining Data with Overlap

There is another data combination situation that can't be expressed as either a merge or concatenation operation. You may have two datasets with indexes that overlap in full or in part. As a motivating example, consider NumPy's ``where`` function, which performs the array-oriented equivalent of an ``if-else`` operation:

In [36]:
a = pd.Series([np.nan, 2.5, 0.0, 3.5, 4.5, np.nan],
            index=["f", "e", "d", "c", "b", "a"])
b = pd.Series([0., np.nan, 2., np.nan, np.nan, 5.],
            index=["a", "b", "c", "d", "e", "f"])

In [37]:
a

f    NaN
e    2.5
d    0.0
c    3.5
b    4.5
a    NaN
dtype: float64

In [38]:
b

a    0.0
b    NaN
c    2.0
d    NaN
e    NaN
f    5.0
dtype: float64

In [ ]:
np.where(pd.isna(a), b, a)

array([0. , 2.5, 0. , 3.5, 4.5, 5. ])

: 